<div dir="rtl">

# מטלת ביניים: ניתוח גישושי לנתוני טויוטה קורולה
**נושא 8, קונה פרטי:** תיבה אוטומטית, האם משלמים עליה?

**מגישים:** ניר אברהמוף, עינב ינון

<div style="text-align:center;">
<img src="https://raw.githubusercontent.com/NirAvrahamoff/toyota-corolla-eda/main/images/cover.jpg" width="620" alt="Iceberg illustration: two Corollas with equal price tags above the water; age, mileage and the automatic gearbox premium hidden below">
<br>
<sub>איור להמחשת הממצא: בהשוואה הגולמית המחירים שווים, ומתחת לפני השטח מסתתרים הגיל, הק״מ והפרמיה על התיבה.</sub>
</div>

</div>

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import statsmodels.formula.api as smf
from openpyxl import load_workbook
from statsmodels.nonparametric.smoothers_lowess import lowess

In [ ]:
FILE = "Toyota_Corolla_cars.xlsx"
toyota_raw = pd.read_excel(FILE, sheet_name="data")

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,   # cleaner frame on every graph
                     "figure.dpi": 110, "axes.titlepad": 10})
TRANS_COLS = {"Manual": "#3E6A8A", "Automatic": "#A3282C"}
ORDER = ["Manual", "Automatic"]
thousands = mtick.FuncFormatter(lambda x, _: f"{x:,.0f}")  # used for EUR and KM axes

FIG_DIR = Path("figures")  # all graphs are saved here
FIG_DIR.mkdir(exist_ok=True)

def save(fig, name):
    """Tight layout, save to figures/, show."""
    fig.tight_layout()
    fig.savefig(FIG_DIR / name, dpi=150)
    plt.show()

# Styled tables: bold caption, dark header row, formatted numbers
TABLE_STYLE = [{"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold"), ("text-align", "left")]},
               {"selector": "th", "props": [("background-color", "#17212B"), ("color", "white"), ("font-weight", "bold")]}]
def styled(df, caption, fmt):
    return df.style.format(fmt).set_caption(caption).set_table_styles(TABLE_STYLE)

print(toyota_raw.shape)

<hr>

<div dir="rtl">

## 1. שאלות המחקר
**הבעיה המחקרית:** קונה פרטי רוצה לדעת אם תיבה אוטומטית עולה יותר בקורולה משומשת, וכמה.

| סוג | שאלה | משתנים בקובץ | מה חסר |
|---|---|---|---|
| תיאור | מהו שיעור הרכבים האוטומטיים לעומת ידניים מכלל הרכבים במדגם, ומהו חציון המחיר של כל קבוצה? | Automatic, Price | |
| קשר | האם יש קשר בין סוג התיבה למחיר, גם כשמשווים רכבים באותו גיל? | Automatic, Price, Age_08_04 | |
|  חיזוי |  בהינתן רכב עם מאפיינים ידועים, מה יהיה מחירו? | Price, Age_08_04, KM, HP, Automatic, Weight, Doors, Fuel_Type | מחיר עסקה בפועל, זמן עד מכירה |
| סיבתיות | האם הפער במחיר הוא תוצאה ישירה של סוג הגיר, או שמדובר בהשפעה עקיפה של משתנים מתערבים אחרים? | Price, Automatic, Age_08_04, KM, HP, Fuel_Type, Airco, Automatic_airco, ABS, Metallic_Rim, Boardcomputer, Mfr_Guarantee, BOVAG_Guarantee | אמינות ותקלות, היסטוריית טיפולים ותאונות, מספר בעלים קודמים, סוג המוכר (סוחר או פרטי), רמת גימור מלאה, מצב הרכב |

</div>

<hr>

<div dir="rtl">

## 2. הכנת הנתונים
העבודה עוברת על שלבי הכנת הנתונים הקלאסיים של פרויקט דאטה. שלב 7 בפרויקט מלא, הכנה למודל (קידוד, Scaling וחלוקה ל־Train/Test), לא נדרש בניתוח גישושי ולא נכלל כאן.

| שלב | מה בודקים | איפה |
|---|---|---|
| 1. הבנת הנתונים | סוגי עמודות (dtypes), תיאור סטטיסטי (describe) | 2.1 |
| 2. ערכים חסרים | isna().sum(), מחיקה או השלמה | 2.2 |
| 3. תיקון סוגי נתונים | המרה למספר או לקטגוריה, ערכים כמו "NA" או "unknown" בתוך טקסט | 2.3 |
| 4. דאטה "מלוכלך" | רווחים, אותיות גדולות וקטנות, שגיאות כתיב, אחידות תאריכים | 2.4 |
| 5. ערכים קיצוניים | זיהוי, והחלטה אם להסיר, לתקן או להשאיר | 2.5 |
| 6. כפילויות | שורות כפולות, כפילויות לפי מזהה | 2.6 |

</div>

<div dir="rtl">

### 2.1 הבנת הנתונים

</div>

In [ ]:
print("Rows:", len(toyota_raw), " Columns:", toyota_raw.shape[1])
print("Column types:", toyota_raw.dtypes.astype(str).value_counts().to_dict())

# Overview of all columns
overview = pd.DataFrame({"dtype": toyota_raw.dtypes.astype(str),
                         "n_unique": toyota_raw.nunique(),
                         "example": toyota_raw.iloc[0]})
display(overview)

# Binary columns (0/1) and numeric non-binary columns, found automatically
binary_cols = [c for c in toyota_raw.columns if set(toyota_raw[c].unique()) <= {0, 1}]
num_cols = [c for c in toyota_raw.select_dtypes("number").columns
            if c != "Id" and c not in binary_cols]
print("Numeric (non-binary) columns:", num_cols)

# describe: min / max reveal implausible values (describe is meaningless for 0/1 columns)
display(toyota_raw[num_cols].describe().round(1))

# Binary columns: share of cars with each feature, in %
display((toyota_raw[binary_cols].mean() * 100).round(1).sort_values().to_frame("% of cars"))

<div dir="rtl">

### 2.2 ערכים חסרים

</div>

In [ ]:
missing = toyota_raw.isna().sum()
print("Missing values (total):", missing.sum())
if missing.any():
    display(missing[missing > 0].to_frame("missing"))
else:
    print("No column has missing values")

<div dir="rtl" style="background-color:#eef3f8; border-right:4px solid #3E6A8A; padding:10px 14px; border-radius:4px;">

**תוצאה:** אין ערכים חסרים באף עמודה, ולכן לא נדרשת מחיקה או השלמה (imputation).

</div>

<div dir="rtl">

### 2.3 בדיקת סוגי נתונים

**הממצא:** כשבדקנו את קובץ האקסל המקורי, לפני שטענו אותו למחברת, ראינו שעמודת `CC` שמורה בו כטקסט ולא כמספר (כל 1,436 התאים).

**למה זה חשוב:** בכלים שלא ממירים סוגים אוטומטית, כמו R או חלק מכלי ה־BI, העמודה תיקרא כטקסט, וחישובים עליה (ממוצע, השוואה, סינון לפי טווח) ייכשלו או יחזירו תוצאה שגויה.

**אצלנו:** `pd.read_excel` מזהה שכל הערכים נראים כמספרים וממיר את העמודה ל־`int64` אוטומטית. לכן בטבלת הסקירה מופיע `int64`, והעמודה תקינה לניתוח.

**ההחלטה:** לא נדרש טיפול. התא הבא מאמת את זה: הוא קורא את סוג התאים ישירות מהקובץ עם `openpyxl` ומשווה לסוג ש־pandas קבע.
    
</div>

In [ ]:
ws = load_workbook(FILE, read_only=True)["data"]
header = [c.value for c in next(ws.iter_rows(max_row=1))]
cc_idx = header.index("CC")
cell_types = pd.Series([type(r[cc_idx]).__name__
                        for r in ws.iter_rows(min_row=2, values_only=True)
                        if r[0] is not None])
print("Cell types in CC column:", cell_types.value_counts().to_dict())
print("pandas dtype after reading:", toyota_raw.CC.dtype)

# Placeholder text that should really be a missing value ("NA", "unknown", "?" ...)
obj_cols = toyota_raw.select_dtypes("object").columns
bad_tokens = {"", "na", "n/a", "nan", "null", "none", "unknown", "?", "-"}
for c in obj_cols:
    s = toyota_raw[c].astype(str).str.strip().str.lower()
    print(f"{c}: placeholder-like values = {s.isin(bad_tokens).sum()}")

<div dir="rtl" style="background-color:#eef3f8; border-right:4px solid #3E6A8A; padding:10px 14px; border-radius:4px;">

**תוצאה:** בקובץ האקסל CC שמורה כטקסט בכל 1,436 התאים. pandas ממיר אותה ל־int64 כבר בקריאה, ולכן היא תקינה כאן; בכלים כמו R צריך להמיר ידנית. בשלב הניקוי מופעל גם `pd.to_numeric` כהגנה, למקרה שהקובץ ייקרא אחרת. בעמודות הטקסט (Model, Fuel_Type, Color) אין ערכים כמו "NA" או "unknown" שמסתירים ערך חסר. Automatic (0/1) מומר לקטגוריה Transmission לצורך הגרפים.

</div>

<div dir="rtl">

### 2.4 דאטה "מלוכלך"

</div>

In [ ]:
# Text: extra spaces, upper/lower case variants
for c in obj_cols:
    raw = toyota_raw[c].astype(str)
    clean = raw.str.strip().str.replace(r"\s+", " ", regex=True).str.lower()
    print(f"{c}: {raw.nunique()} unique values, {clean.nunique()} after strip/lower/single spaces")
# Model: where do the extra variants come from?
m = toyota_raw.Model.astype(str)
print("Model rows with leading/trailing spaces (incl. non-breaking \\xa0):", (m != m.str.strip()).sum())
model_clean = m.str.strip().str.replace(r"\s+", " ", regex=True).str.upper()
print("Model unique: raw", m.nunique(), "| after strip", m.str.strip().nunique(),
      "| after strip + upper case", model_clean.nunique())
ms = m.str.strip().drop_duplicates()
case_var = ms.groupby(ms.str.upper()).agg(list)
display(case_var[case_var.str.len() > 1].head(3).to_frame("case variants"))
print("Fuel_Type values:", toyota_raw.Fuel_Type.value_counts().to_dict())

# Dates: is Age_08_04 consistent with the manufacturing month and year?
if {"Mfg_Month", "Mfg_Year"} <= set(toyota_raw.columns):
    age_calc = (2004 - toyota_raw.Mfg_Year) * 12 + (8 - toyota_raw.Mfg_Month)
    print("Age_08_04 minus age from Mfg_Year/Mfg_Month:",
          (toyota_raw.Age_08_04 - age_calc).value_counts().to_dict())

<div dir="rtl" style="background-color:#eef3f8; border-right:4px solid #3E6A8A; padding:10px 14px; border-radius:4px;">

**תוצאה:** בעמודה Model יש 150 ערכים עם רווח מוביל, רובם רווח לא־שביר (`\xa0`), וגם הבדלי אותיות גדולות וקטנות (למשל Liftback מול LIFTBACK). אחרי ניקוי הרווחים ואיחוד האותיות מספר הדגמים יורד מ־372 ל־312. מנקים את Model כי בדיקת הכפילויות נשענת עליו, וגם זיהוי סוג האוטומט (AUT3, AUT4) ורמת הגימור נלקחים ממנו. Fuel_Type ו־Color נקיות. התאריכים עקביים: בכל השורות Age_08_04 גדול בחודש אחד בדיוק מהגיל שמחושב מ־Mfg_Year ו־Mfg_Month. זה הפרש קבוע של מוסכמת ספירה, לא טעות.

</div>

<div dir="rtl">

### 2.5 ערכים קיצוניים וחריגים

</div>

In [ ]:
# Issue 1: impossible engine size
display(toyota_raw.loc[toyota_raw.CC > 2000, ["Id", "Model", "CC", "HP", "Price"]])
# Issue 2: almost-zero KM
display(toyota_raw.loc[toyota_raw.KM < 100, ["Id", "KM", "Age_08_04", "Price"]])
# Issue 3: constant column
print("Unique values of Cylinders:", toyota_raw.Cylinders.unique())
# Issue 4: price outliers by the IQR rule
q1, q3 = toyota_raw.Price.quantile([.25, .75])
upper = q3 + 1.5 * (q3 - q1)
lower = q1 - 1.5 * (q3 - q1)
print("Price IQR fences:", lower, "-", upper, " cars below:", (toyota_raw.Price < lower).sum(),
      " cars above:", (toyota_raw.Price > upper).sum())
# Issue 5: Weight and HP (both enter the prediction model)
display(toyota_raw[["Weight", "HP"]].describe().round(0).T)
# all heavy cars: Verso (MPV) and diesels are really heavy, small petrol cars 223 and 964 are not
display(toyota_raw.loc[toyota_raw.Weight > 1300, ["Id", "Model", "Weight", "HP", "Fuel_Type", "Price"]])

# Extent of each issue (count and % of all cars)
n = len(toyota_raw)
def pct(k):
    return f"{k} cars ({k / n:.1%})"
print("Price outliers above the fence:", pct((toyota_raw.Price > upper).sum()))
print("KM below 100:", pct((toyota_raw.KM < 100).sum()), "| KM = 1 and older than 24 months:",
      pct(((toyota_raw.KM == 1) & (toyota_raw.Age_08_04 > 24)).sum()))

# Issue 6: near-constant and duplicated columns
for c in ["Parking_Assistant", "Power_Steering"]:
    print(f"{c}: cars with 1 = {toyota_raw[c].sum()} ({toyota_raw[c].mean():.1%})")
print("Radio vs Radio_cassette: rows that differ =", (toyota_raw.Radio != toyota_raw.Radio_cassette).sum())

# Issue 7: is Gears consistent with the transmission type?
display(pd.crosstab(toyota_raw.Gears, toyota_raw.Automatic.map({0: "Manual", 1: "Automatic"})))
# The model name says which gearbox it is: AUT3 / AUT4 = classic 3- or 4-speed automatic
model_up = toyota_raw.Model.str.upper()
aut_type = np.select([model_up.str.contains("AUT4"), model_up.str.contains("AUT3"), model_up.str.contains("AUT")],
                     ["AUT4", "AUT3", "AUT (no number)"], "no AUT in name")
display(pd.crosstab(aut_type, toyota_raw.Automatic.map({0: "Manual", 1: "Automatic"}), rownames=["Model name"]))
print("Gears recorded for AUT3/AUT4 cars:",
      toyota_raw.loc[np.isin(aut_type, ["AUT3", "AUT4"]), "Gears"].value_counts().to_dict())

# Issue 8: bivariate outliers - price far from the price expected for the car's age
fit = np.polyfit(toyota_raw.Age_08_04, toyota_raw.Price, 2)
resid = toyota_raw.Price - np.polyval(fit, toyota_raw.Age_08_04)
biv = toyota_raw.loc[resid.abs() > 3 * resid.std(), ["Id", "Model", "Age_08_04", "Price", "Automatic"]]
print("Bivariate outliers (price more than 3 SD from the age trend):", pct(len(biv)))
display(biv.assign(Model=biv.Model.str.strip()).sort_values("Price", ascending=False))
is_verso = toyota_raw.Model.str.contains("VERSO", case=False)
print("Verso (MPV) cars:", is_verso.sum(), "| of them automatic:", toyota_raw.loc[is_verso, "Automatic"].sum())

In [ ]:
high = toyota_raw[toyota_raw.Price > upper]
print(f"Above the fence - median age: {high.Age_08_04.median():.0f} months (all cars: {toyota_raw.Age_08_04.median():.0f}), "
      f"age <= 36 months: {(high.Age_08_04 <= 36).sum()} of {len(high)}, "
      f"median KM: {high.KM.median():,.0f} (all cars: {toyota_raw.KM.median():,.0f})")

<div dir="rtl">

### 2.6 כפילויות

</div>

In [ ]:
print("Duplicate rows (all columns except Id):", toyota_raw.drop(columns="Id").duplicated().sum())
print("Duplicate Id values:", toyota_raw.Id.duplicated().sum())
same_car = ["Model", "Price", "Age_08_04", "KM", "HP"]
print("Same car listed twice (same model, price, age, KM, HP):",
      toyota_raw.assign(Model=model_clean).duplicated(subset=same_car).sum())
dup = toyota_raw.assign(Model=model_clean).duplicated(subset=same_car, keep=False)
display(toyota_raw.loc[dup, ["Id"] + same_car + ["Color", "Fuel_Type", "Automatic", "Doors", "Mfg_Year"]])
pair = toyota_raw.loc[dup]
print("Columns that differ between the two rows:",
      [c for c in toyota_raw.columns if pair[c].nunique() > 1])
print("Id range:", toyota_raw.Id.min(), "-", toyota_raw.Id.max(),
      "-> missing Ids:", toyota_raw.Id.max() - len(toyota_raw))

<div dir="rtl" style="background-color:#faf3f3; border-right:4px solid #A3282C; padding:10px 14px; border-radius:4px;">

### יומן בעיות והחלטות
| בעיה | מיקום | היקף | החלטה ונימוק |
|---|---|---|---|
| ערכים חסרים | כל העמודות | 0 | אין צורך בהשלמה |
| נפח מנוע לא אפשרי | CC, רכב 81 | רכב אחד (0.1%) | שגיאת הקלדה: שם הדגם מציין 1.6 ליטר ו־110 כ"ס מתאימים למנוע כזה. מתקנים ל־1,600. |
| רווחים ואותיות בשם הדגם | Model | 150 רכבים (10.4%) עם רווח מוביל, רובו `\xa0` | מנקים רווחים ומאחדים לאותיות גדולות: 372 דגמים הופכים ל־312. |
| אותו רכב פעמיים | רכבים 114 ו־115 | זוג אחד | זהים בכל העמודות חוץ מ־Id ורווח בשם, כנראה מודעה כפולה. מוחקים את 115 (רכב ידני, לא משפיע על הפרמיה). |
| ק"מ כמעט אפס | KM | 9 רכבים (0.6%) מתחת ל־100 ק"מ | 7 מהם צעירים מ־10 חודשים, כנראה רכבי תצוגה. סביר, משאירים. |
| ק"מ = 1 ברכב ישן (חוסר מוסווה) | רכבים 605 ו־1442, בני 50 ו־76 חודשים | 2 רכבים (0.1%) | כנראה ערך חסר שקודד כ־1. שניהם ידניים, ולכן מסמנים ב־`km_suspect` במקום להחליף ב־NA, ובודקים שהפרמיה לא משתנה בלעדיהם: בטבלת החציונים (סעיף 4) השינוי הוא 2.5 € בקבוצה אחת, וברגרסיה המלאה (סעיף 5) הפרמיה היא 600 € במקום 602 €. |
| משקל לא סביר | Weight, רכבים 223 ו־964 | 2 רכבים (0.1%): בנזין קטן עם 1,615 ו־1,480 ק"ג, מול חציון 1,060 ק"ג ברכבי בנזין. ה־Verso שוקל גם הוא 1,480, אבל הוא MPV ומשקלו אמיתי | כנראה טעות, אבל הערך הנכון לא ידוע. משאירים ומסמנים ב־`weight_suspect` (בנזין מעל 1,300 ק"ג). |
| משתנה קבוע | Cylinders | כל הרכבים = 4 | אין שונות, מוחקים. |
| משתנים כמעט קבועים וכפולים | Parking_Assistant, Power_Steering, Radio ו־Radio_cassette | חיישני חניה ב־4 רכבים (0.3%), הגה כוח ב־97.8%, Radio ו־Radio_cassette שונים רק ב־3 רכבים | כמעט אין בהם מידע. לא נכנסים למודל, ומהזוג Radio ו־Radio_cassette משאירים רק את Radio. |
| חריגי מחיר | Price | 110 רכבים (7.7%) מעל 17,200 €, אף רכב מתחת ל־3,200 € | רכבים צעירים ויקרים אמיתיים, משאירים. |
</div>

In [ ]:
toyota = toyota_raw.copy()
toyota["Model"] = model_clean
toyota = toyota[~toyota.duplicated(subset=same_car)].copy()  # same car listed twice
toyota["CC"] = pd.to_numeric(toyota.CC)
toyota.loc[toyota.CC > 3000, "CC"] = 1600
toyota["km_suspect"] = (toyota.KM < 100) & (toyota.Age_08_04 > 24)
toyota["weight_suspect"] = (toyota.Weight > 1300) & (toyota.Fuel_Type == "Petrol")  # Id 223 and 964: small petrol cars with MPV weight
toyota["km_per_month"] = toyota.KM / toyota.Age_08_04  # derived: how much the car is driven
toyota["Transmission"] = pd.Categorical(
    np.where(toyota.Automatic == 1, "Automatic", "Manual"), categories=ORDER)
toyota["Age_band"] = pd.cut(toyota.Age_08_04, [0, 24, 48, 60, 70, 80],
                            labels=["0-24", "25-48", "49-60", "61-70", "71-80"])
toyota["KM_band"] = pd.cut(toyota.KM, [-1, 50000, 100000, 150000, np.inf],
                           labels=["0-50K", "50-100K", "100-150K", "150K+"])
toyota = toyota.drop(columns="Cylinders")
print("Suspicious KM flagged:", toyota.km_suspect.sum(), "| Suspicious weight flagged:", toyota.weight_suspect.sum())

<hr>

<div dir="rtl">

## 3. שאלת התיאור

</div>

In [ ]:
print(toyota.Transmission.value_counts())
print((toyota.Transmission.value_counts(normalize=True) * 100).round(1))

desc = toyota.groupby("Transmission", observed=True).agg(
    n=("Price", "size"), median_price=("Price", "median"),
    mean_price=("Price", "mean"), median_age=("Age_08_04", "median"),
    median_km=("KM", "median"), median_hp=("HP", "median")).round(0)
display(styled(desc, "Description by transmission type",
               {"n": "{:,.0f}", "median_price": "{:,.0f} €", "mean_price": "{:,.0f} €", "median_age": "{:.0f} months",
                "median_km": "{:,.0f} km", "median_hp": "{:.0f} hp"}))
display(pd.crosstab(toyota.HP, toyota.Transmission))
display(pd.crosstab(toyota.Fuel_Type, toyota.Transmission))

<div dir="rtl" style="background-color:#eef3f8; border-right:4px solid #3E6A8A; padding:10px 14px; border-radius:4px;">

**תובנה:** ההספק וסוג הדלק קשורים לתיבה. מנוע 107 כ"ס מופיע רק באוטומטיים, ואין אף אוטומט בדיזל (שתי הטבלאות האחרונות). לכן ההשוואה ההוגנת בסעיף 4 מסוננת לבנזין 86 עד 110 כ"ס, וגם שם לקבוצת ה־107 כ"ס אין מקבילה ידנית.

</div>

<div dir="rtl">

**גרף 1:** תרשים קופסה של המחיר לפי סוג התיבה, לכל הרכבים במדגם. הקו בתוך הקופסה הוא החציון, היהלום הלבן הוא הממוצע והנקודות הן חריגים. הגרף עונה על שאלת התיאור: האם יש הבדל במחיר בין הקבוצות בלי שום בקרה.

</div>

In [ ]:
# Graph 1: raw price comparison (looks like no difference)
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=toyota, x="Transmission", y="Price", order=ORDER,
            hue="Transmission", palette=TRANS_COLS, width=0.5,
            boxprops=dict(alpha=0.7), flierprops=dict(alpha=0.4), ax=ax)
means = toyota.groupby("Transmission", observed=True).Price.mean()
ax.scatter(range(2), means[ORDER], marker="D", color="white",
           edgecolor="black", s=60, zorder=5)
ax.yaxis.set_major_formatter(thousands)
ax.set(xlabel=None, ylabel="Price (EUR)")
ax.set_title("Do automatics cost more? (raw comparison, no controls)\n"
             f"Median: manual {desc.median_price['Manual']:,.0f} EUR, "
             f"automatic {desc.median_price['Automatic']:,.0f} EUR "
             f"(n = {desc.n['Manual']:.0f} vs {desc.n['Automatic']:.0f})",
             loc="left", fontsize=11)
save(fig, "g1_raw_price.png")

<div dir="rtl" style="background-color:#eef3f8; border-right:4px solid #3E6A8A; padding:10px 14px; border-radius:4px;">

**תובנה:** בהשוואה הגולמית חציון המחיר כמעט זהה (ידני 9,900 €, אוטומט 9,950 €), כך שבמבט ראשון התיבה האוטומטית לא עולה כלום.

</div>

<div dir="rtl">

**גרף 2:** התפלגות הגיל (בחודשים) של הרכבים הידניים והאוטומטיים, כעקומות צפיפות (KDE). הגרף בודק אם שתי הקבוצות דומות בגיל, ולפניו מודפס המתאם בין המחיר לגיל.

</div>

In [ ]:
print("r(Price, Age) =", round(toyota.Price.corr(toyota.Age_08_04), 3))

# Graph 2: automatics are older (the hidden third variable)
fig, ax = plt.subplots(figsize=(7, 5))
for t in ORDER:
    sns.kdeplot(toyota.loc[toyota.Transmission == t, "Age_08_04"], fill=True, clip=(0, 80),
                alpha=0.45, color=TRANS_COLS[t], label=t, ax=ax)
    med = toyota.loc[toyota.Transmission == t, "Age_08_04"].median()
    ax.axvline(med, color=TRANS_COLS[t], ls="--", lw=1.6, label=f"{t} median: {med:.0f} months")
ax.set(xlabel="Age (months, Aug 2004)", ylabel="Density")
ax.set_title("Are automatics older than manuals?\n"
             f"Median age: manual {desc.median_age['Manual']:.0f} months, "
             f"automatic {desc.median_age['Automatic']:.0f} months",
             loc="left", fontsize=11)
ax.legend()
save(fig, "g2_age_density.png")

<div dir="rtl" style="background-color:#eef3f8; border-right:4px solid #3E6A8A; padding:10px 14px; border-radius:4px;">

**תובנה:** האוטומטיים מבוגרים יותר מהידניים (חציון 65 חודשים לעומת 60), והגיל הוא המשתנה החזק ביותר למחיר (r = −0.88), ולכן הוא יכול להסתיר פער.

</div>

<hr>

<div dir="rtl">

## 4. שאלת הקשר: בקרה על גיל (משתנה שלישי)

</div>

In [ ]:
def premium_table(df, band):
    t = (df.groupby([band, "Transmission"], observed=True).Price.median()
           .unstack())
    t["premium"] = t["Automatic"] - t["Manual"]
    t["premium_pct"] = (t["premium"] / t["Manual"] * 100).round(1)  # premium as % of the manual median
    return t

by_age_n = toyota.groupby(["Age_band", "Transmission"], observed=True).size().unstack(fill_value=0)
age_wide = premium_table(toyota, "Age_band")
display(by_age_n)
display(styled(age_wide, "Median price and automatic premium by age band",
               {"Manual": "{:,.0f} €", "Automatic": "{:,.0f} €", "premium": "{:+,.0f} €", "premium_pct": "{:+.1f}%"})
        .background_gradient(subset=["premium_pct"], cmap="Reds"))

<div dir="rtl">

**גרף 3 (בדיקת משתנה שלישי):** תרשים קופסה של המחיר לפי סוג התיבה, בנפרד לכל אחת מחמש קבוצות הגיל. המספרים באדום הם הפרש החציונים (אוטומט פחות ידני) בכל קבוצה ומספר האוטומטיים בה.

</div>

In [ ]:
# Graph 3 (key graph): within every age band, automatic is more expensive
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=toyota, x="Age_band", y="Price", hue="Transmission",
            hue_order=ORDER, palette=TRANS_COLS, width=0.7,
            boxprops=dict(alpha=0.7), flierprops=dict(alpha=0.3), ax=ax)
n_auto = by_age_n.loc[age_wide.index, "Automatic"]
for i, (p, pct, n) in enumerate(zip(age_wide.premium, age_wide.premium_pct, n_auto)):
    ax.text(i, 34300, f"{p:+,.0f} EUR ({pct:+.1f}%)\n(n auto = {n})", ha="center", va="bottom",
            color="#A3282C", fontweight="bold", fontsize=9)
ax.set_ylim(top=38500)
ax.yaxis.set_major_formatter(thousands)
ax.set(xlabel="Age band (months)", ylabel="Price (EUR)")
ax.set_title("At the same age, do automatics cost more?\n"
             "Red labels: median premium (EUR, % of manual median); n auto = automatics in band",
             loc="left", fontsize=11)
ax.legend(title=None, loc="center right")
save(fig, "g3_price_by_age_band.png")

<div dir="rtl" style="background-color:#eef3f8; border-right:4px solid #3E6A8A; padding:10px 14px; border-radius:4px;">

**תובנה:** ברכב באותו גיל האוטומט יקר יותר בכל חמש הקבוצות, בפער חציוני של 455 עד 1,625 €, שהם 4.3% עד 11.6% מהמחיר, וברוב הקבוצות כ־8%.

ביורו הפרמיה יורדת עם הגיל, אבל באחוזים היא כמעט קבועה, כך שהירידה ביורו נובעת בעיקר מכך שרכבים ישנים זולים יותר. הפרמיה הוסתרה בהשוואה הגולמית כי האוטומטיים מבוגרים יותר. בקבוצה הצעירה יש רק 6 אוטומטיים, ולכן הפער שם לא מדויק (רווחי סמך בהמשך).

</div>

<div dir="rtl">

**כמה יציבה הפרמיה?** בחלק מהקבוצות החציון של האוטומטיים נשען על מעט רכבים, ולכן מחשבים רווח סמך 95% ב־bootstrap להפרש החציונים.

</div>

In [ ]:
def bootstrap_premium(df, band, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for b, g in df.groupby(band, observed=True):
        man = g.loc[g.Transmission == "Manual", "Price"].to_numpy()
        aut = g.loc[g.Transmission == "Automatic", "Price"].to_numpy()
        if len(man) == 0 or len(aut) == 0:
            continue
        diffs = (np.median(rng.choice(aut, (n_boot, len(aut))), axis=1)
                 - np.median(rng.choice(man, (n_boot, len(man))), axis=1))
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        rows.append({band: b, "n_auto": len(aut), "n_manual": len(man),
                     "premium": np.median(aut) - np.median(man),
                     "ci_low": lo, "ci_high": hi})
    return pd.DataFrame(rows).set_index(band).round(0)

display(bootstrap_premium(toyota, "Age_band"))

<div dir="rtl" style="background-color:#eef3f8; border-right:4px solid #3E6A8A; padding:10px 14px; border-radius:4px;">

**תובנה:** בארבע מחמש הקבוצות רווח הסמך כולו מעל אפס. בקבוצה הצעירה (0 עד 24) יש רק 6 אוטומטיים, ולכן הרווח רחב (550 עד 2,705 €). בקבוצת 25 עד 48 הרווח כולל את האפס (‎−255 עד 3,550 €), כך שהפרמיה שם לא יציבה. הכיוון עקבי בכל הקבוצות, אבל הגודל בכל קבוצה בנפרד לא מדויק.

</div>

<div dir="rtl">

**בדיקות רובוסטיות:** רק רכבי בנזין בטווח 86 עד 110 כ"ס (המנועים שבאמת מגיעים עם אוטומט), ובלי הק"מ החשוד.

</div>

In [ ]:
fair = toyota[(toyota.Fuel_Type == "Petrol") & toyota.HP.between(86, 110)]
print("Petrol, 86-110 HP:"); display(premium_table(fair, "Age_band"))
print("Without suspicious KM:"); display(premium_table(toyota[~toyota.km_suspect], "Age_band"))

<div dir="rtl">

**רמת גימור ואחריות:** בקובץ יש משתני ציוד ואחריות. אם לאוטומטיים יש יותר ציוד או יותר אחריות, ייתכן שהפרמיה היא עליהם ולא על התיבה. הציוד קשור גם לגיל (למשל מחשב דרך), והאוטומטיים מבוגרים יותר, ולכן משווים גם בתוך קבוצות גיל (`diff_within_age_pp`, ממוצע משוקלל לפי מספר האוטומטיים בכל קבוצה).

</div>

In [ ]:
equip_cols = ["ABS", "Airbag_1", "Airbag_2", "Airco", "Automatic_airco", "Boardcomputer",
              "CD_Player", "Central_Lock", "Powered_Windows", "Power_Steering", "Radio",
              "Mistlamps", "Sport_Model", "Metallic_Rim", "Met_Color", "Tow_Bar"]
# Hebrew translation of the equipment and warranty columns
HEB = {"ABS": "בלמי ABS", "Airbag_1": "כרית אוויר לנהג", "Airbag_2": "כרית אוויר לנוסע",
       "Airco": "מזגן", "Automatic_airco": "מזגן אוטומטי (בקרת אקלים)", "Boardcomputer": "מחשב נסיעה",
       "CD_Player": "נגן דיסקים", "Central_Lock": "נעילה מרכזית", "Powered_Windows": "חלונות חשמליים",
       "Power_Steering": "הגה כוח", "Radio": "רדיו", "Mistlamps": "פנסי ערפל",
       "Sport_Model": "חבילת ספורט", "Metallic_Rim": "חישוקי סגסוגת (ג׳אנטים)", "Met_Color": "צבע מטאלי",
       "Tow_Bar": "וו גרירה", "Mfr_Guarantee": "אחריות יצרן", "BOVAG_Guarantee": "אחריות BOVAG (סוחרים מורשים)"}
# Equipment and warranty (warranty can also raise the price)
check_cols = [c for c in equip_cols + ["Mfr_Guarantee", "BOVAG_Guarantee"] if c in toyota.columns]
equip = (toyota.groupby("Transmission", observed=True)[check_cols].mean().T * 100).round(1)
equip["diff_pp"] = equip["Automatic"] - equip["Manual"]

# Equipment is linked to age (e.g. Boardcomputer), and automatics are older,
# so we also compare within age bands (weighted by the number of automatics in each band)
w = by_age_n["Automatic"]
def within_age_diff(col):
    g = toyota.groupby(["Age_band", "Transmission"], observed=True)[col].mean().unstack()
    return ((g["Automatic"] - g["Manual"]) * w).sum() / w.sum() * 100
equip["diff_within_age_pp"] = [round(within_age_diff(c), 1) for c in equip.index]
equip.insert(0, "Hebrew", equip.index.map(HEB))  # translation as the first column
display(equip.sort_values("diff_within_age_pp"))
print("Guarantee_Period (months), median:",
      toyota.groupby("Transmission", observed=True).Guarantee_Period.median().to_dict())

<div dir="rtl" style="background-color:#eef3f8; border-right:4px solid #3E6A8A; padding:10px 14px; border-radius:4px;">

**תובנה:** גם בהשוואה בתוך קבוצות גיל אין לאוטומטיים יתרון בציוד, וההפרש הגדול ביותר הוא דווקא לטובת הידניים (חישוקי מתכת, ‎−13.5 נקודות אחוז).

בתוך קבוצות הגיל הפער במחשב דרך מצטמצם מ־‎−7.4 ל־‎−2.6 נקודות אחוז, כי הוא קשור לגיל. היתרונות הבולטים של האוטומטיים הם מזגן אוטומטי (+6.5) ואחריות יצרן (+7.1 נקודות אחוז). אחריות יכולה להעלות את המחיר, ולכן היא נכנסת כמשתנה בקרה ברגרסיה בסעיף 5.

</div>

<hr>

<div dir="rtl">

## 5. הרחבה מעבר לדרישות הקורס
בדיקות נוספות לשאלת הסיבתיות: האם הפרמיה נשארת כשמבקרים על ק״מ, על מנוע ועל כמה משתנים יחד.

**בקרה על ק״מ:** האם הפרמיה נשארת כשמשווים רכבים בתוך אותה קבוצת ק״מ?

</div>

In [ ]:
display(toyota.groupby(["KM_band", "Transmission"], observed=True).size().unstack(fill_value=0))
display(premium_table(toyota, "KM_band"))
print("Median KM by transmission:")
print(toyota.groupby("Transmission", observed=True).KM.median())
print("Median age (months) by KM band:")
display(toyota.groupby(["KM_band", "Transmission"], observed=True).Age_08_04.median().unstack())

<div dir="rtl">

**גרף 4:** פיזור של המחיר מול הגיל, רק לרכבי בנזין עם מנוע של 86 עד 110 כ"ס (המנועים שמגיעים גם עם אוטומט), עם קו מגמה ליניארי לכל סוג תיבה. מתחת לגרף מודפס הפער בין הקווים בגיל 24, 48 ו־80 חודשים.

</div>

In [ ]:
# Graph 4: price vs age with a linear line per transmission (comparable engines)
fig, ax = plt.subplots(figsize=(8, 5))
rng = np.random.default_rng(1)
fits = {}  # intercept and slope per transmission
for t in ORDER:
    s = fair[fair.Transmission == t]
    ax.scatter(s.Age_08_04 + rng.uniform(-0.4, 0.4, len(s)), s.Price,
               alpha=0.25, s=12, color=TRANS_COLS[t], label=t)
    b, a = np.polyfit(s.Age_08_04, s.Price, 1)
    xs = np.array([s.Age_08_04.min(), s.Age_08_04.max()])
    ax.plot(xs, a + b * xs, color=TRANS_COLS[t], lw=2.2)
    fits[t] = (a, b)
ax.yaxis.set_major_formatter(thousands)
ax.set(xlabel="Age (months)", ylabel="Price (EUR)")
ax.set_title("Does the premium hold for comparable engines, at every age?\n"
             "Petrol cars, 86-110 HP only", loc="left", fontsize=11)
ax.legend()
save(fig, "g4_price_age_lines.png")

# Gap between the two lines at a few ages
for age in [24, 48, 80]:
    gap = (fits["Automatic"][0] + fits["Automatic"][1] * age) - (fits["Manual"][0] + fits["Manual"][1] * age)
    print(f"Gap (automatic - manual) at age {age} months: {gap:,.0f} EUR")

<div dir="rtl" style="background-color:#eef3f8; border-right:4px solid #3E6A8A; padding:10px 14px; border-radius:4px;">

**תובנה:** גם בין רכבי בנזין עם מנוע דומה (86 עד 110 כ"ס) האוטומט יקר יותר בכל גיל, והפער בין הקווים יורד מכ־1,700 € בגיל 24 חודשים לכ־530 € בגיל 80 חודשים.

כלומר הפרמיה לא מוסברת בהבדלי מנוע או דלק. ביורו היא קטנה ככל שהרכב מתיישן, אבל ביחס למחיר הירידה קטנה בהרבה (גרף 3). ברגרסיה עם אינטראקציה בין תיבה לגיל (בהמשך הסעיף) הירידה באותו כיוון אבל לא מובהקת (p = 0.41), ולכן זו מגמה ולא ממצא מוכח.

</div>

<div dir="rtl">

**גרף 5:** פיזור של הק"מ מול הגיל לכל הרכבים, עם עקומת LOWESS (קו מגמה מוחלק) לכל סוג תיבה. הגרף בודק אם האוטומטיים נוסעים פחות מהידניים באותו גיל.

</div>

In [ ]:
# Graph 5: the complication - automatics are older but driven less
fig, ax = plt.subplots(figsize=(8, 5))
for t, a, sz in [("Manual", 0.15, 10), ("Automatic", 0.8, 22)]:
    s = toyota[toyota.Transmission == t]
    ax.scatter(s.Age_08_04, s.KM, alpha=a, s=sz, color=TRANS_COLS[t], label=t)
    lo = lowess(s.KM, s.Age_08_04, frac=0.6)
    ax.plot(lo[:, 0], lo[:, 1], color=TRANS_COLS[t], lw=2.2)
ax.yaxis.set_major_formatter(thousands)
ax.set(xlabel="Age (months)", ylabel="KM")
ax.set_title("At the same age, are automatics driven less?\n"
             "KM vs. age with LOWESS curve, by transmission type", loc="left", fontsize=11)
ax.legend()
save(fig, "g5_age_km.png")
print("Median KM per month:", toyota.groupby("Transmission", observed=True).km_per_month.median().round(0).to_dict())

<div dir="rtl" style="background-color:#eef3f8; border-right:4px solid #3E6A8A; padding:10px 14px; border-radius:4px;">

**תובנה:** אוטומט נוסע בחציון 931 ק"מ בחודש לעומת 1,143 בידני, כ־19% פחות, ואין אף אוטומט מעל 150 אלף ק"מ.

עד גיל של כ־30 חודשים שתי הקבוצות נוסעות בערך אותו דבר, והפער נפתח מגיל זה ומעלה. לכן ייתכן שחלק מהפרמיה הוא על הק"מ הנמוך ולא על התיבה. הבדיקה בהמשך מפרידה בין השניים.

</div>

<div dir="rtl">

**בקרה על כמה משתנים יחד:** בתוך כל קבוצת ק"מ האוטומטיים שוב מבוגרים יותר (טבלת הגיל החציוני למעלה), כך שפיצול לפי ק"מ בלבד מערבב שוב את הגיל. לכן בודקים ברגרסיה שמבקרת על גיל וק"מ יחד (OLS, שגיאות תקן חסינות HC3). זו הצצה ראשונה לשיעור 4, לא מודל סופי.

**גרף 6:** הפרמיה של האוטומט (מקדם הרגרסיה) עם רווח סמך 95%, כשמוסיפים בכל שורה עוד משתני בקרה: בלי בקרה, גיל, ק"מ, הספק ודלק, ציוד ואחריות. הקו המקווקו הוא אפס, כלומר אין פרמיה.

</div>

In [ ]:
models = {
    "nothing": "Price ~ Automatic",
    "age": "Price ~ Automatic + Age_08_04",
    "age + km": "Price ~ Automatic + Age_08_04 + KM",
    "age + km + hp + fuel": "Price ~ Automatic + Age_08_04 + KM + HP + C(Fuel_Type, Treatment('Petrol'))",
    "+ equipment + warranty": ("Price ~ Automatic + Age_08_04 + KM + HP + C(Fuel_Type, Treatment('Petrol'))"
                    " + Airco + Automatic_airco + ABS + Metallic_Rim + Boardcomputer"
                    " + Mfr_Guarantee + BOVAG_Guarantee"),
}
rows = []
for name, formula in models.items():
    m = smf.ols(formula, data=toyota).fit(cov_type="HC3")
    lo, hi = m.conf_int().loc["Automatic"]
    rows.append({"controls": name, "premium": m.params["Automatic"], "ci_low": lo,
                 "ci_high": hi, "p_value": m.pvalues["Automatic"], "R2": m.rsquared})
reg = pd.DataFrame(rows).set_index("controls")
display(styled(reg, "Automatic premium as controls are added (OLS, HC3 standard errors)",
               {"premium": "{:,.0f} €", "ci_low": "{:,.0f} €", "ci_high": "{:,.0f} €", "p_value": "{:.4f}", "R2": "{:.3f}"}))


# Graph 6: how the premium changes as controls are added
fig, ax = plt.subplots(figsize=(8, 4.8))
labels = ["No controls", "+ Age", "+ Age + KM", "+ Age + KM + HP + fuel", "+ equipment + warranty"]
ypos = np.arange(len(reg))[::-1]
for y, (name, row) in zip(ypos, reg.iterrows()):
    c = "#5A6672" if name == "nothing" else TRANS_COLS["Automatic"]
    ax.plot([row.ci_low, row.ci_high], [y, y], color=c, lw=3)
    ax.plot(row.premium, y, "o", color=c, ms=10)
    ax.text(row.ci_high + 50, y, f"{row.premium:,.0f} EUR", va="center", fontweight="bold")
ax.axvline(0, color="black", lw=1, ls="--")
ax.set_yticks(ypos, labels)
ax.xaxis.set_major_formatter(thousands)
ax.set_xlim(right=reg.ci_high.max() + 450)
ax.set(xlabel="Automatic premium (EUR), 95% CI", ylabel=None)
ax.set_title("Does the automatic premium survive when we control for other variables?\n"
             "OLS with HC3 standard errors", loc="left", fontsize=11)
save(fig, "g6_premium_by_controls.png")

# Does the premium shrink with age? (interaction between transmission and age)
inter = smf.ols("Price ~ Automatic * Age_08_04 + KM + HP + C(Fuel_Type, Treatment('Petrol'))", data=toyota).fit(cov_type="HC3")
terms = ["Automatic", "Automatic:Age_08_04"]
print("Interaction model:")
display(pd.DataFrame({"coef": inter.params[terms], "p_value": inter.pvalues[terms]}).round(4))
for age in [24, 48, 80]:
    print(f"  premium at age {age} months: {inter.params['Automatic'] + inter.params['Automatic:Age_08_04'] * age:,.0f} EUR")

# Premium in percent (log price)
logm = smf.ols("np.log(Price) ~ Automatic + Age_08_04 + KM + HP + C(Fuel_Type, Treatment('Petrol'))", data=toyota).fit(cov_type="HC3")
lo, hi = logm.conf_int().loc["Automatic"]
print(f"Log-price model: premium = {np.expm1(logm.params['Automatic']):.1%} "
      f"(95% CI {np.expm1(lo):.1%} to {np.expm1(hi):.1%})")

# Sensitivity: full model without the 2 cars with KM = 1 (treated as missing, as in class)
m_sens = smf.ols(models["+ equipment + warranty"], data=toyota[~toyota.km_suspect]).fit(cov_type="HC3")
lo, hi = m_sens.conf_int().loc["Automatic"]
print(f"Full model without the 2 suspicious-KM cars: premium = {m_sens.params['Automatic']:,.0f} EUR "
      f"(95% CI {lo:,.0f} to {hi:,.0f}), n = {int(m_sens.nobs)}")

# Sensitivity: all 16 equipment variables instead of 5 (the choice of equipment does not drive the result)
f_all = (models["age + km + hp + fuel"] + " + " + " + ".join(equip_cols)
         + " + Mfr_Guarantee + BOVAG_Guarantee")
m_all = smf.ols(f_all, data=toyota).fit(cov_type="HC3")
lo, hi = m_all.conf_int().loc["Automatic"]
print(f"Full model with all 16 equipment variables: premium = {m_all.params['Automatic']:,.0f} EUR "
      f"(95% CI {lo:,.0f} to {hi:,.0f})")

# Sensitivity: + trim level, taken from the model name (Terra = base, Luna = mid, Sol = high)
trim = np.select([toyota.Model.str.contains(t) for t in ["TERRA", "LUNA", "SOL"]],
                 ["Terra", "Luna", "Sol"], "Other")
m_trim = smf.ols(f_all + " + C(trim)", data=toyota.assign(trim=trim)).fit(cov_type="HC3")
lo, hi = m_trim.conf_int().loc["Automatic"]
print(f"... + trim level from the model name: premium = {m_trim.params['Automatic']:,.0f} EUR "
      f"(95% CI {lo:,.0f} to {hi:,.0f})")
print("Trim level by transmission:")
display(pd.crosstab(trim, toyota.Transmission, rownames=["Trim"]))

<div dir="rtl" style="background-color:#eef3f8; border-right:4px solid #3E6A8A; padding:10px 14px; border-radius:4px;">

**תובנה:** כשמבקרים ברגרסיה על גיל וק"מ יחד נשארת פרמיה של 712 € (רווח סמך 408 עד 1,016 €), ואחרי בקרה גם על הספק, דלק, ציוד ואחריות נשארת פרמיה של 602 € (321 עד 884 €).

כשמפצלים לפי ק"מ הפרמיה נראית לא עקבית, אבל זה אותו ערבוב שראינו בגיל: בתוך כל קבוצת ק"מ האוטומטיים מבוגרים יותר (למשל 47 מול 41 חודשים בקבוצת 0 עד 50 אלף), והגיל מסתיר שוב את הפרמיה.

**הפרמיה באחוזים:** במודל עם לוגריתם המחיר האוטומט יקר ב־7.1% (רווח סמך 4.6% עד 9.7%) מרכב ידני עם אותו גיל, ק"מ, הספק ודלק.

**האם הפרמיה קטנה עם הגיל?** במודל עם אינטראקציה הפרמיה המוערכת יורדת מכ־1,140 € בגיל 24 חודשים לכ־650 € בגיל 80, אבל האינטראקציה לא מובהקת (p = 0.41). עם 80 אוטומטיים בלבד אין מספיק נתונים כדי לקבוע זאת, ובאחוזים מהמחיר הפרמיה כמעט קבועה (גרף 3), כך שחלק גדול מהירידה ביורו נובע מכך שרכבים ישנים זולים יותר.

**רגישות לק״מ החשוד:** בלי שני הרכבים עם ק״מ = 1 (כאילו הם ערך חסר, כמו בשיעור), המודל המלא נותן פרמיה של 600 € (רווח סמך 316 עד 883 €), כמעט זהה ל־602 €.

**בחירת משתני הציוד ורמת גימור:** עם כל 16 משתני הציוד, במקום חמישה, הפרמיה היא 593 € (רווח סמך 317 עד 870 €). כשמוסיפים גם רמת גימור מתוך שם הדגם (Terra, Luna, Sol) היא 591 € (312 עד 871 €). כלומר הבחירה של משתני הציוד לא משנה את התוצאה, וגם רמת הגימור לא מסבירה את הפרמיה.

**חציון מול ממוצע:** בסעיף 4 הפרמיה היא הפרש חציונים בתוך קבוצת גיל, וכאן היא מקדם רגרסיה, כלומר הפרש ממוצעים אחרי בקרה. לכן המספרים לא זהים, אבל שתי השיטות מראות פרמיה חיובית באותו סדר גודל.

**השערה:** חלק מהפרמיה מוסבר בק"מ הנמוך, אבל לא כולה, והיא נשארת חיובית אחרי כל הבקרות. לא ברור אם KM הוא משתנה מתערב או משתנה מתווך (מי שקונה אוטומט נוסע פחות, ולכן הרכב שומר על ערכו). אם הוא מתווך, הבקרה עליו מורידה חלק מהאפקט של התיבה. לכן הפרמיה מוערכת בין כ־600 ל־970 €, כ־7% מהמחיר. זו הערכה מתוך נתוני תצפית ולא הוכחה סיבתית.

</div>

<hr>

<div dir="rtl">

## 6. השערות לחיזוי ולסיבתיות
במטלה הזו שתי השאלות האלה נשארות השערות. לכל אחת: על מה היא נשענת, ומה נדרש כדי לבדוק אותה.

<table dir="rtl" style="width:100%; direction:rtl;">
<tr><th style="text-align:right; vertical-align:top; padding:6px 10px;">סוג</th><th style="text-align:right; vertical-align:top; padding:6px 10px;">השערה</th><th style="text-align:right; vertical-align:top; padding:6px 10px;">על סמך</th><th style="text-align:right; vertical-align:top; padding:6px 10px;">נדרש כדי לבדוק</th></tr>
<tr><td style="text-align:right; vertical-align:top; padding:6px 10px;"><b>חיזוי</b></td><td style="text-align:right; vertical-align:top; padding:6px 10px;">גיל, ק״מ, הספק, משקל, דלתות ודלק יחזו את המחיר היטב, והתיבה תשפר את הדיוק רק מעט.</td><td style="text-align:right; vertical-align:top; padding:6px 10px;">הגיל לבדו מתואם ‎−0.88 עם המחיר, והרגרסיות בסעיף 5 מסבירות 83% עד 87% מהשונות (על כל הנתונים, לא על Test). רק 5.6% מהרכבים אוטומטיים, ולכן התיבה נוגעת למעט רכבים.</td><td style="text-align:right; vertical-align:top; padding:6px 10px;">מודל שנבנה על Train ונבדק על Test, עם התיבה ובלעדיה, והשוואת שגיאת החיזוי.</td></tr>
<tr><td style="text-align:right; vertical-align:top; padding:6px 10px;"><b>סיבתיות</b></td><td style="text-align:right; vertical-align:top; padding:6px 10px;">חלק מהפער נובע מהתיבה עצמה, בערך 7% מהמחיר (כ־600 עד 970 €).</td><td style="text-align:right; vertical-align:top; padding:6px 10px;">הפער נשאר באותו גיל (גרף 3), בין מנועים דומים (גרף 4), ואחרי בקרה על ק״מ, הספק, דלק, ציוד, אחריות ורמת גימור (סעיף 5).</td><td style="text-align:right; vertical-align:top; padding:6px 10px;">נתונים על סוג המוכר, תאונות ומספר בעלים. בלעדיהם אי אפשר לשלול מתערבים שלא נמדדו.</td></tr>
</table>

## 7. מה הנתונים לא מאפשרים לדעת
| מגבלה | מה זה משנה במסקנה |
|---|---|
| רק 80 אוטומטיים, ובקבוצת הגיל הצעירה רק 6. | רווחי הסמך רחבים: הפרמיה ברכב צעיר, והירידה שלה עם הגיל, לא מדויקות. |
| מחיר מבוקש ולא מחיר עסקה. | ייתכן שהפרמיה קיימת רק במודעה ונשחקת במשא ומתן, כך שהקונה עשוי לשלם בפועל פחות ממנה. |
| מנוע 107 כ"ס קיים רק באוטומט, ואין אוטומט בדיזל. | את השפעת המנוע הזה אי אפשר להפריד מהתיבה, והממצא לא חל על קונה שמחפש דיזל. |
| מתערבים שלא נמדדו: סוג המוכר (סוחר או פרטי), היסטוריית תאונות וטיפולים, מספר בעלים, מצב הרכב. | אם, למשל, אוטומטיים נמכרים יותר אצל סוחרים שגובים יותר, חלק מהפרמיה הוא על המוכר ולא על התיבה. לכן אי אפשר לקבוע שהתיבה עצמה גורמת לפער. |
| אין זמן עד מכירה. | לא ידוע אם אוטומט נמכר מהר יותר, ולכן אי אפשר לומר אם השוק באמת מוכן לשלם את הפרמיה. |
| קורולה בלבד, הולנד, 2004. | בשוק שבו אוטומט נפוץ יותר, או בדגמים אחרים, הפרמיה עשויה להיות שונה לגמרי. |

## 8. סיכום לבעל העניין

**מה הייתי אומר לקונה:** באותו גיל, אוטומט ידרשו בכ־7% יותר, כלומר כ־600 עד 970 € ברכב במחיר חציוני (לפי הרגרסיה בסעיף 5; בגרף 3 הפער החציוני הוא כ־8% ברוב קבוצות הגיל). השוו רק רכבים באותו גיל.
</div>